# Chapter 7 - Audiovisual Synchronization with Cross-Modal Fusion

Companion notebook for *Video Generation with AI* (O'Reilly) by Joseph Enochs.

**Learning objective:** Understand how to generate synchronized audio and video simultaneously with cross-modal fusion, by building the key pieces of the LTX-2 pipeline: mel-spectrogram audio representation, the unified DiT transformer with bidirectional audio-video attention, the audio VAE and vocoder path, and two-stage flow matching inference.

<a href="https://colab.research.google.com/github/jenochs/video-ai-book-companion/blob/main/chapter-07/notebooks/chapter_7_companion_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Every cell below runs top to bottom on a free Colab CPU. Audio examples use small synthetic signals, so there are no large downloads. Loading the real 19B+ LTX-2 checkpoints is opt-in and clearly marked near the end.

## Setup

Install the few packages we need. Colab ships with recent PyTorch, so this is quick.

In [ ]:
%pip install -q torch torchaudio einops matplotlib numpy

In [ ]:
import math
from dataclasses import dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch import Tensor

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | torchaudio {torchaudio.__version__} | device: {device}")

## The Missing Modality: Audio

Chapters 3-5 generated silent video. Sound has to be generated jointly with video because human perception demands alignment at multiple scales: frame-level (a door slam on the exact frame), phoneme-level (lips within 40-80 ms of speech sounds), semantic, and ambient. The numbers below show why post-hoc audio is so hard: viewers detect lip-sync errors of about 45 ms, roughly two frames at 24 FPS.

In [ ]:
# The synchronization precision problem, in numbers.
fps = 24
sample_rate = 16_000
clip_seconds = 5

video_frames = fps * clip_seconds
audio_samples = sample_rate * clip_seconds
samples_per_frame = audio_samples / video_frames

lip_sync_threshold_ms = 45
frames_at_threshold = lip_sync_threshold_ms / (1000 / fps)

print(f"A {clip_seconds}s clip: {video_frames} video frames vs {audio_samples:,} audio samples")
print(f"That is {samples_per_frame:.0f} audio samples per video frame")
print(f"Humans detect lip-sync errors of ~{lip_sync_threshold_ms} ms "
      f"= {frames_at_threshold:.1f} frames at {fps} FPS")

The fix is joint generation: produce both modalities simultaneously, with architectural connections that enforce synchronization instead of trying to recover timing from ambiguous visual cues afterward.

## Audio Representation: From Waveforms to Mel-Spectrograms

Video is 4D and raw audio is 1D at 650+ samples per frame, so LTX-2 models audio as mel-spectrograms: a 2D map of frequency content over time, like sheet music for machines. The `AudioProcessor` below is the actual transformation from the LTX-2 codebase.

In [ ]:
# From ltx_core/model/audio_vae/ops.py - AudioProcessor
class AudioProcessor(nn.Module):
    def __init__(self, sample_rate: int, mel_bins: int, mel_hop_length: int, n_fft: int):
        super().__init__()
        self.sample_rate = sample_rate
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,         # 16000 Hz
            n_fft=n_fft,                     # 1024
            win_length=n_fft,                # 1024
            hop_length=mel_hop_length,       # 160 samples
            f_min=0.0,
            f_max=sample_rate / 2.0,         # 8000 Hz Nyquist
            n_mels=mel_bins,                 # 64 mel bins
            mel_scale="slaney",
        )

    def resample_waveform(self, waveform, orig_rate, target_rate):
        if orig_rate == target_rate:
            return waveform
        return torchaudio.functional.resample(waveform, orig_rate, target_rate)

    def waveform_to_mel(self, waveform, waveform_sample_rate):
        # Resample if needed, then compute log-mel spectrogram
        waveform = self.resample_waveform(waveform, waveform_sample_rate, self.sample_rate)
        mel = self.mel_transform(waveform)
        mel = torch.log(torch.clamp(mel, min=1e-5))
        return mel.permute(0, 1, 3, 2)  # [B, C, time, freq]

audio_processor = AudioProcessor(sample_rate=16_000, mel_bins=64, mel_hop_length=160, n_fft=1024)
print("Mel frames per second:", 16_000 / 160)

We test it on a synthetic scene: a low hum (ambience), a rising chirp, and two sharp clicks at 0.5 s and 1.5 s. Think of the clicks as door slams, the kind of event that must land on an exact video frame.

In [ ]:
def make_synthetic_scene_audio(duration_sec=2.0, sample_rate=16_000):
    """Hum + rising chirp + two clicks, shaped [1, 1, samples]."""
    n = int(duration_sec * sample_rate)
    t = torch.arange(n) / sample_rate

    hum = 0.20 * torch.sin(2 * math.pi * 110.0 * t)
    f0, f1 = 300.0, 3000.0
    chirp = 0.15 * torch.sin(2 * math.pi * (f0 * t + (f1 - f0) * t**2 / (2 * duration_sec)))

    clicks = torch.zeros(n)
    for click_time in (0.5, 1.5):
        start = int(click_time * sample_rate)
        length = int(0.01 * sample_rate)
        envelope = torch.exp(-torch.arange(length) / (0.002 * sample_rate))
        clicks[start:start + length] += 0.8 * envelope

    return (hum + chirp + clicks).clamp(-1.0, 1.0).reshape(1, 1, -1)

waveform = make_synthetic_scene_audio()
mel = audio_processor.waveform_to_mel(waveform, 16_000)
print("Waveform:", tuple(waveform.shape), "-> Mel:", tuple(mel.shape), "[B, C, time, freq]")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
time_axis = torch.arange(waveform.shape[-1]) / 16_000
axes[0].plot(time_axis, waveform[0, 0], linewidth=0.5)
axes[0].set_ylabel("Amplitude")
axes[0].set_title("Raw waveform (1D, 32,000 samples)")
axes[1].imshow(mel[0, 0].T, aspect="auto", origin="lower",
               extent=[0, 2.0, 0, 64], cmap="magma")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Mel bin")
axes[1].set_title("Log-mel spectrogram (2D, 64 bins x ~100 frames/s)")
for ax in axes:
    for click_time in (0.5, 1.5):
        ax.axvline(click_time, color="cyan", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

from IPython.display import Audio, display
display(Audio(waveform[0, 0].numpy(), rate=16_000))

The clicks show up as sharp vertical lines: exactly the temporal precision the transformer needs for synchronization. Phase is discarded, which is fine because the human ear barely notices it, and a neural vocoder reconstructs it at synthesis time.

## Introducing LTX-2

LTX-2 processes both modalities through a single 48-layer transformer. Each layer has a larger video stream and a smaller audio stream joined by bidirectional cross-modal attention, so synchronization emerges from training on paired data rather than from explicit alignment supervision.

In [ ]:
# LTX-2.3 key specifications (Table 7-2 in the chapter)
ltx2_specs = {
    "production_version": "LTX-2.3 (Lightricks/LTX-2.3)",
    "full_checkpoint_parameters": "22B (transformer + VAEs + vocoder + upsampler + projections)",
    "transformer_parameters": "19B = 14B video + 5B audio (asymmetric dual-stream)",
    "transformer_layers": 48,
    "video_stream_inner_dim": 4096,   # 32 heads x 128 d_head
    "audio_stream_inner_dim": 2048,   # 32 heads x 64 d_head
    "cross_modal_attention": "bidirectional, video<->audio at every layer",
    "text_encoder": "Gemma 3 (12B), separate caption projections per stream",
    "video_latent_channels": 128,     # 3D VAE: 32x spatial + 8x temporal compression
    "audio_latent_channels": 128,     # Mel-spectrogram VAE, 4x temporal compression
    "audio_sample_rate": "16 kHz encoder, 24 kHz vocoder output",
    "frame_count_constraint": "frames % 8 == 1 (1, 9, 17, ..., 121)",
}
for key, value in ltx2_specs.items():
    print(f"{key:28s} {value}")

generation_modes = ["Text-to-audio-video (T2AV)", "Image-to-audio-video (I2AV)",
                    "Video-to-video (IC-LoRA)", "Keyframe interpolation"]
print("\nGeneration modes:", ", ".join(generation_modes))

## Unified DiT Architecture

Unlike twin-backbone designs that fuse separate networks at a few points, LTX-2 keeps video and audio coupled inside every transformer block. We now build a runnable miniature of that architecture, starting with the `Modality` dataclass that carries each stream's latents, timesteps, positions, and text context.

In [ ]:
# From ltx_core/model/transformer/modality.py
@dataclass(frozen=True)
class Modality:
    enabled: bool                 # Whether this modality is active
    latent: Tensor                # [B, seq_len, channels]
    timesteps: Tensor             # [B, seq_len] per-token timesteps
    positions: Tensor             # [B, dims, seq_len, 2] position bounds
    context: Tensor               # Text embeddings for this modality
    context_mask: Tensor | None = None

# Frozen means immutable: updates create new instances via dataclasses.replace(),
# which keeps the forward pass free of hidden state changes.

### AdaLN: Adaptive Layer Normalization

Each block learns a `scale_shift_table` of six vectors per stream: shift, scale, and gate for attention, and again for the feedforward. The gates let layers contribute more or less as denoising moves from coarse structure to fine detail.

In [ ]:
class AdaLNModulation(nn.Module):
    """Timestep conditioning: produces shift, scale, gate for MSA and MLP."""
    def __init__(self, dim):
        super().__init__()
        # AdaLN applies shift, scale, and gate to attention and FFN outputs
        self.scale_shift_table = nn.Parameter(torch.randn(6, dim) / dim**0.5)
        self.timestep_proj = nn.Sequential(nn.Linear(dim, dim), nn.SiLU(), nn.Linear(dim, 6 * dim))

    def forward(self, timestep_embedding):
        b = timestep_embedding.shape[0]
        emb = self.timestep_proj(timestep_embedding).view(b, 6, -1)
        # [shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp]
        return tuple((self.scale_shift_table[None] + emb).unsqueeze(2).chunk(6, dim=1))


def timestep_embedding(timesteps, dim, max_period=10_000):
    """Standard sinusoidal timestep embedding, shape [B, dim]."""
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32) / half)
    args = timesteps.float()[:, None] * freqs[None].to(timesteps.device)
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)

# Quick check: six modulation tensors per stream
demo = AdaLNModulation(dim=64)
mods = demo(timestep_embedding(torch.tensor([500.0]), 64))
print("AdaLN produces", len(mods), "modulation tensors, each shaped", tuple(mods[0].shape))

### The BasicAVTransformerBlock

The heart of synchronization. Each block runs, in order: self-attention per modality, text cross-attention per modality, then bidirectional cross-modal attention on the text-conditioned states, then feedforward. Audio queries attend to video keys and values while video queries attend to audio.

In [ ]:
# Modeled on ltx_core/model/transformer/transformer.py - BasicAVTransformerBlock,
# scaled down so it runs on CPU (production: video dim 4096, audio dim 2048, 48 layers).
class BasicAVTransformerBlock(nn.Module):
    def __init__(self, video_dim=128, audio_dim=64, num_heads=4, text_dim=96, ff_mult=4):
        super().__init__()

        def attn(q_dim, kv_dim):
            return nn.MultiheadAttention(embed_dim=q_dim, num_heads=num_heads,
                                         kdim=kv_dim, vdim=kv_dim, batch_first=True)

        def ff(dim):
            return nn.Sequential(nn.Linear(dim, dim * ff_mult), nn.GELU(),
                                 nn.Linear(dim * ff_mult, dim))

        # Video path components
        self.attn1 = attn(video_dim, video_dim)                # Video self-attention
        self.attn2 = attn(video_dim, text_dim)                 # Video cross-attention (text)
        self.audio_to_video_attn = attn(video_dim, audio_dim)  # Audio -> Video cross-attn
        self.ff = ff(video_dim)
        self.video_adaln = AdaLNModulation(video_dim)          # AdaLN for video
        self.norm_v1 = nn.LayerNorm(video_dim, elementwise_affine=False)
        self.norm_v2, self.norm_v3 = nn.LayerNorm(video_dim), nn.LayerNorm(video_dim)
        self.norm_v_ff = nn.LayerNorm(video_dim, elementwise_affine=False)

        # Audio path components
        self.audio_attn1 = attn(audio_dim, audio_dim)          # Audio self-attention
        self.audio_attn2 = attn(audio_dim, text_dim)           # Audio cross-attention (text)
        self.video_to_audio_attn = attn(audio_dim, video_dim)  # Video -> Audio cross-attn
        self.audio_ff = ff(audio_dim)
        self.audio_adaln = AdaLNModulation(audio_dim)          # AdaLN for audio
        self.norm_a1 = nn.LayerNorm(audio_dim, elementwise_affine=False)
        self.norm_a2, self.norm_a3 = nn.LayerNorm(audio_dim), nn.LayerNorm(audio_dim)
        self.norm_a_ff = nn.LayerNorm(audio_dim, elementwise_affine=False)

    @staticmethod
    def _modulated(norm, x, shift, scale):
        return norm(x) * (1 + scale.squeeze(1)) + shift.squeeze(1)

    def forward(self, video, audio, video_context, audio_context, video_t_emb, audio_t_emb):
        vs, vsc, vg, vs2, vsc2, vg2 = self.video_adaln(video_t_emb)
        as_, asc, ag, as2, asc2, ag2 = self.audio_adaln(audio_t_emb)

        # 1. Self-attention with AdaLN modulation (video and audio in parallel)
        v_in = self._modulated(self.norm_v1, video, vs, vsc)
        video = video + vg.squeeze(1) * self.attn1(v_in, v_in, v_in)[0]
        a_in = self._modulated(self.norm_a1, audio, as_, asc)
        audio = audio + ag.squeeze(1) * self.audio_attn1(a_in, a_in, a_in)[0]

        # 2. Text cross-attention for both
        video = video + self.attn2(self.norm_v2(video), video_context, video_context)[0]
        audio = audio + self.audio_attn2(self.norm_a2(audio), audio_context, audio_context)[0]

        # 3. Bidirectional cross-modal attention (on text-conditioned states)
        video_kv, audio_kv = self.norm_v3(video), self.norm_a3(audio)
        # Audio attends to video (audio queries, video keys/values)
        audio = audio + self.video_to_audio_attn(audio_kv, video_kv, video_kv)[0]
        # Video attends to audio (video queries, audio keys/values)
        video = video + self.audio_to_video_attn(video_kv, audio_kv, audio_kv)[0]

        # 4. Feedforward networks with AdaLN modulation
        video = video + vg2.squeeze(1) * self.ff(self._modulated(self.norm_v_ff, video, vs2, vsc2))
        audio = audio + ag2.squeeze(1) * self.audio_ff(self._modulated(self.norm_a_ff, audio, as2, asc2))
        return video, audio

print("BasicAVTransformerBlock defined: two parallel paths, four attention types.")

### A miniature LTXModel

Stacking blocks with input and output projections gives the full model shape: both modalities enter, both are denoised together, both exit as predictions.

In [ ]:
# Modeled on ltx_core/model/transformer/model.py - LTXModel
class MiniAVTransformer(nn.Module):
    def __init__(self, num_layers=2, video_dim=128, audio_dim=64,
                 latent_channels=128, text_dim=96, num_heads=4):
        super().__init__()
        self.video_dim, self.audio_dim = video_dim, audio_dim

        # Input processing (patchify projections)
        self.video_proj_in = nn.Linear(latent_channels, video_dim)
        self.audio_proj_in = nn.Linear(latent_channels, audio_dim)

        # Unified transformer blocks for both modalities
        self.transformer_blocks = nn.ModuleList([
            BasicAVTransformerBlock(video_dim=video_dim, audio_dim=audio_dim,
                                    num_heads=num_heads, text_dim=text_dim)
            for _ in range(num_layers)
        ])

        # Output processing
        self.video_norm_out, self.audio_norm_out = nn.LayerNorm(video_dim), nn.LayerNorm(audio_dim)
        self.video_proj_out = nn.Linear(video_dim, latent_channels)
        self.audio_proj_out = nn.Linear(audio_dim, latent_channels)

    def forward(self, video: Modality, audio: Modality):
        v = self.video_proj_in(video.latent)
        a = self.audio_proj_in(audio.latent)
        v_t = timestep_embedding(video.timesteps.float().mean(dim=1), self.video_dim)
        a_t = timestep_embedding(audio.timesteps.float().mean(dim=1), self.audio_dim)
        for block in self.transformer_blocks:
            v, a = block(v, a, video.context, audio.context, v_t, a_t)
        return (self.video_proj_out(self.video_norm_out(v)),
                self.audio_proj_out(self.audio_norm_out(a)))

mini_model = MiniAVTransformer(num_layers=2)

# Build the two Modality objects, exactly as in the chapter's forward pass
B, video_tokens, audio_tokens = 1, 24, 12
video = Modality(enabled=True, latent=torch.randn(B, video_tokens, 128),
                 timesteps=torch.full((B, video_tokens), 500.0),
                 positions=torch.zeros(B, 3, video_tokens, 2),   # F, H, W bounds
                 context=torch.randn(B, 8, 96))
audio = Modality(enabled=True, latent=torch.randn(B, audio_tokens, 128),
                 timesteps=torch.full((B, audio_tokens), 500.0),
                 positions=torch.zeros(B, 1, audio_tokens, 2),   # time bounds
                 context=torch.randn(B, 8, 96))

# Forward returns predictions for both
video_pred, audio_pred = mini_model(video=video, audio=audio)
print("video_pred:", tuple(video_pred.shape), "| audio_pred:", tuple(audio_pred.shape))

total = sum(p.numel() for p in mini_model.parameters())
print(f"Mini model parameters: {total:,} (production LTX-2: 19B transformer)")

To see the bidirectional coupling in action, we perturb only the audio input and check that the video prediction changes. In a twin-backbone model with late fusion this influence would be much weaker; here it flows through every layer.

In [ ]:
with torch.no_grad():
    video_pred_1, _ = mini_model(video=video, audio=audio)

    # Same video input, different audio input (frozen dataclass: use replace)
    audio_perturbed = replace(audio, latent=audio.latent + torch.randn_like(audio.latent))
    video_pred_2, _ = mini_model(video=video, audio=audio_perturbed)

difference = (video_pred_1 - video_pred_2).abs().mean().item()
print(f"Mean change in VIDEO prediction from perturbing only AUDIO input: {difference:.4f}")
print("Nonzero: audio influences video at every layer through audio_to_video_attn.")

## Audio Pipeline Deep Dive

The full audio path runs waveform -> mel-spectrogram -> VAE latents (4x temporal compression, 128 channels) -> transformer -> decoded mel -> vocoder -> 24 kHz waveform. We already built the mel front end; the next pieces are latent normalization, timing, and synthesis.

### Per-Channel Statistics Normalization

Each of the 128 latent channels is standardized with dataset-wide statistics, like calibrating each channel of a mixing board. This keeps noise scales consistent for diffusion and makes cross-modal attention with similarly normalized video latents behave well.

In [ ]:
# From ltx_core/model/audio_vae/ops.py
class PerChannelStatistics(nn.Module):
    """Per-channel statistics computed over the training dataset."""

    def __init__(self, latent_channels: int = 128):
        super().__init__()
        # In LTX-2 these buffers are loaded from the checkpoint
        # (names match checkpoint keys, hence the hyphens).
        self.register_buffer("std-of-means", torch.empty(latent_channels))
        self.register_buffer("mean-of-means", torch.empty(latent_channels))

    def normalize(self, x):
        # Standardize: (x - mean) / std
        return (x - self.get_buffer("mean-of-means")) / self.get_buffer("std-of-means")

    def un_normalize(self, x):
        # Inverse: x * std + mean
        return (x * self.get_buffer("std-of-means")) + self.get_buffer("mean-of-means")

stats = PerChannelStatistics(latent_channels=128)
# Simulate checkpoint statistics for the demo
stats.get_buffer("mean-of-means").copy_(torch.randn(128) * 0.5)
stats.get_buffer("std-of-means").copy_(torch.rand(128) * 2 + 0.5)

fake_latent = torch.randn(4, 128) * 3 + 1
normalized = stats.normalize(fake_latent)
roundtrip = stats.un_normalize(normalized)
print(f"Round trip max error: {(roundtrip - fake_latent).abs().max().item():.2e}")

### AudioPatchifier: tokens with timecodes

Audio uses `patch_size=1`: frequency bins become feature channels, preserving full frequency resolution per latent frame. The patchifier also maps latent indices to real seconds, the timing metadata that lets audio events align to video frames. One audio latent frame is about 40 ms, roughly one video latent frame at 25 fps.

In [ ]:
# From ltx_core/components/patchifiers.py - AudioPatchifier (core logic)
class AudioPatchifier:
    def __init__(self, patch_size=1, sample_rate=16_000, hop_length=160,
                 audio_latent_downsample_factor=4, is_causal=True):
        self.sample_rate = sample_rate
        self.hop_length = hop_length
        self.audio_latent_downsample_factor = audio_latent_downsample_factor
        self.is_causal = is_causal

    def patchify(self, audio_latents):
        # [B, C, T, F] -> [B, T, C*F] - flatten freq into features
        b, c, t, f = audio_latents.shape
        return audio_latents.permute(0, 2, 1, 3).reshape(b, t, c * f)

    def unpatchify(self, audio_tokens, channels, mel_bins):
        # [B, T, C*F] -> [B, C, T, F] - restore structure
        b, t, _ = audio_tokens.shape
        return audio_tokens.reshape(b, t, channels, mel_bins).permute(0, 2, 1, 3)

    def latent_frame_times_sec(self, start_latent, end_latent):
        # Convert latent indices to real-time seconds
        audio_latent_frame = torch.arange(start_latent, end_latent)
        audio_mel_frame = audio_latent_frame * self.audio_latent_downsample_factor
        if self.is_causal:
            # Offset for causal alignment
            audio_mel_frame = (audio_mel_frame + 1 - self.audio_latent_downsample_factor).clip(min=0)
        return audio_mel_frame * self.hop_length / self.sample_rate

patchifier = AudioPatchifier()
audio_latents = torch.randn(1, 128, 50, 1)   # [B, C, T, F] after VAE encoding
tokens = patchifier.patchify(audio_latents)
restored = patchifier.unpatchify(tokens, channels=128, mel_bins=1)
print("Latents", tuple(audio_latents.shape), "-> tokens", tuple(tokens.shape),
      "| lossless round trip:", torch.equal(restored, audio_latents))

times = patchifier.latent_frame_times_sec(0, 8)
print("First 8 latent frames start at (seconds):", [f"{t:.3f}" for t in times.tolist()])
print("Spacing after warmup: 40 ms per latent frame (4 mel frames x 10 ms hop)")

### HiFi-GAN Style Vocoder

The vocoder turns decoded mel-spectrograms back into waveforms with 240x upsampling: mel arrives at ~100 frames/s and the output needs 24,000 samples/s, so 24,000 / 100 = 240 = 6 x 5 x 2 x 2 x 2 across five transposed-convolution stages. Multi-Receptive Field fusion averages parallel ResBlocks with different kernel sizes to capture patterns at several temporal scales.

In [ ]:
# Modeled on ltx_core/model/audio_vae/vocoder.py, at reduced channel width for CPU.
LRELU_SLOPE = 0.1

class ResBlock(nn.Module):
    """Dilated residual block (dilations 1, 3, 5) at one kernel size."""
    def __init__(self, channels, kernel_size, dilations=(1, 3, 5)):
        super().__init__()
        self.convs1 = nn.ModuleList([
            nn.Conv1d(channels, channels, kernel_size, dilation=d,
                      padding=(kernel_size - 1) * d // 2) for d in dilations])
        self.convs2 = nn.ModuleList([
            nn.Conv1d(channels, channels, kernel_size,
                      padding=(kernel_size - 1) // 2) for _ in dilations])

    def forward(self, x):
        for c1, c2 in zip(self.convs1, self.convs2):
            x = x + c2(F.leaky_relu(c1(F.leaky_relu(x, LRELU_SLOPE)), LRELU_SLOPE))
        return x


class Vocoder(nn.Module):
    def __init__(self,
                 resblock_kernel_sizes=[3, 7, 11],
                 upsample_rates=[6, 5, 2, 2, 2],          # Total: 240x
                 upsample_kernel_sizes=[16, 15, 8, 4, 4],
                 upsample_initial_channel=128,             # Production: 1024
                 stereo=True, mel_bins=64):
        super().__init__()
        self.num_upsamples = len(upsample_rates)
        self.num_kernels = len(resblock_kernel_sizes)

        # Input: 128 channels for stereo (2 x 64 mel bins)
        in_channels = 2 * mel_bins if stereo else mel_bins
        self.conv_pre = nn.Conv1d(in_channels, upsample_initial_channel, 7, padding=3)

        # 5 upsampling stages with transposed convolutions
        self.ups = nn.ModuleList()
        for i, (stride, kernel_size) in enumerate(zip(upsample_rates, upsample_kernel_sizes)):
            self.ups.append(nn.ConvTranspose1d(
                upsample_initial_channel // (2**i),
                upsample_initial_channel // (2**(i + 1)),
                kernel_size, stride, padding=(kernel_size - stride) // 2))

        # Multi-receptive field fusion ResBlocks: 3 kernels x 5 stages = 15 blocks
        self.resblocks = nn.ModuleList()
        for i in range(self.num_upsamples):
            channels = upsample_initial_channel // (2**(i + 1))
            for kernel_size in resblock_kernel_sizes:
                self.resblocks.append(ResBlock(channels, kernel_size))

        final_channels = upsample_initial_channel // (2**self.num_upsamples)
        # Output: 2 channels for stereo waveform
        self.conv_post = nn.Conv1d(final_channels, 2 if stereo else 1, 7, padding=3)
        self.upsample_factor = math.prod(upsample_rates)   # 240

    def forward(self, x):
        # Rearrange stereo mel: [B, 2, T, 64] -> [B, 128, T]
        b, s, t, f = x.shape
        x = x.permute(0, 1, 3, 2).reshape(b, s * f, t)
        x = self.conv_pre(x)
        for i in range(self.num_upsamples):
            x = self.ups[i](F.leaky_relu(x, LRELU_SLOPE))
            # Apply parallel ResBlocks with different receptive fields, then average
            start = i * self.num_kernels
            outs = torch.stack([self.resblocks[j](x) for j in range(start, start + self.num_kernels)])
            x = outs.mean(dim=0)
        return torch.tanh(self.conv_post(F.leaky_relu(x)))   # [B, 2, T*240]

vocoder = Vocoder()
mel_input = torch.randn(1, 2, 24, 64)          # 24 mel frames of stereo audio
with torch.no_grad():
    waveform_out = vocoder(mel_input)
print(f"Mel {tuple(mel_input.shape)} -> waveform {tuple(waveform_out.shape)}")
print(f"Upsample factor: {vocoder.upsample_factor}x "
      f"({mel_input.shape[2]} frames -> {waveform_out.shape[-1]} samples)")
print("Note: weights are untrained here, so the output is noise; the point is the shapes.")

## Two-Stage Inference Pipeline

LTX-2 first paints structure, then refines detail: Stage 1 runs 40 CFG-guided Euler steps at half resolution, a learned upsampler doubles the resolution, and Stage 2 runs just 4 steps with a distilled LoRA and no CFG. We build every piece with a toy model so the whole flow runs in seconds.

### Classifier-Free Guidance and the Euler step

CFG amplifies what makes the prompted prediction different from the unprompted one. The Euler stepper moves the latent along the model's predicted velocity toward the clean sample.

In [ ]:
# From ltx_core/components/guiders.py
@dataclass(frozen=True)
class CFGGuider:
    """Classifier-free guidance guider."""
    scale: float = 4.0   # LTX-2 Stage 1 default

    def delta(self, cond, uncond):
        # Amplify difference between positive and negative
        return (self.scale - 1) * (cond - uncond)

    def enabled(self):
        return self.scale != 1.0

    def apply(self, cond, uncond):
        # output = cond + scale_delta, i.e. pos + (scale-1)*(pos - neg)
        return cond + self.delta(cond, uncond) if self.enabled() else cond


class EulerDiffusionStep:
    """x_{t+1} = x_t + (sigma_next - sigma_now) * velocity."""
    def step(self, latent, denoised, sigmas, step_index):
        sigma_now, sigma_next = sigmas[step_index], sigmas[step_index + 1]
        if float(sigma_now) == 0.0:
            return denoised
        velocity = (latent - denoised) / sigma_now
        return latent + (sigma_next - sigma_now) * velocity

guider = CFGGuider(scale=4.0)
print("CFG enabled:", guider.enabled(), "| scale:", guider.scale)

### The joint Euler denoising loop

Both modalities are updated together at every step. That co-updating is what preserves audio-video alignment while noise is removed.

In [ ]:
# From ltx_pipelines/utils/helpers.py - euler_denoising_loop (simplified)
@dataclass(frozen=True)
class LatentState:
    latent: Tensor
    denoise_mask: Tensor | None = None
    clean_latent: Tensor | None = None

def post_process_latent(denoised, denoise_mask, clean_latent):
    # Blend with clean latent where conditioning applies
    if denoise_mask is None or clean_latent is None:
        return denoised
    return denoise_mask * denoised + (1 - denoise_mask) * clean_latent

def euler_denoising_loop(sigmas, video_state, audio_state, stepper, denoise_fn):
    """Joint audio-video denoising over a diffusion schedule."""
    history = []
    for step_idx in range(len(sigmas) - 1):
        # Get denoised predictions from model (both modalities at once)
        denoised_video, denoised_audio = denoise_fn(video_state, audio_state, sigmas, step_idx)

        denoised_video = post_process_latent(denoised_video, video_state.denoise_mask,
                                             video_state.clean_latent)
        denoised_audio = post_process_latent(denoised_audio, audio_state.denoise_mask,
                                             audio_state.clean_latent)

        # Euler step for both modalities together
        video_state = replace(video_state,
            latent=stepper.step(video_state.latent, denoised_video, sigmas, step_idx))
        audio_state = replace(audio_state,
            latent=stepper.step(audio_state.latent, denoised_audio, sigmas, step_idx))
        history.append((video_state.latent.clone(), audio_state.latent.clone()))
    return video_state, audio_state, history

print("Joint denoising loop defined.")

### Running the toy two-stage pipeline

Our toy "model" pulls latents toward a pair of correlated targets: a video pattern with two bright events and an audio pattern with energy bursts at the same times, standing in for on-screen actions and their sounds. Stage 1 uses 40 steps with CFG; Stage 2 upsamples 2x and refines with the 4 distilled sigmas, no CFG.

In [ ]:
# Correlated targets: video events at positions 1/4 and 3/4, audio bursts at the same times
video_len, audio_len = 32, 16
t_video = torch.linspace(0, 1, video_len)
t_audio = torch.linspace(0, 1, audio_len)
video_target = (torch.exp(-((t_video - 0.25) / 0.05)**2) +
                torch.exp(-((t_video - 0.75) / 0.05)**2)).reshape(1, video_len, 1)
audio_target = (torch.exp(-((t_audio - 0.25) / 0.05)**2) +
                torch.exp(-((t_audio - 0.75) / 0.05)**2)).reshape(1, audio_len, 1)

def make_toy_denoiser(v_target, a_target, guider):
    """An x0-model stand-in: predicts targets when 'prompted', zeros when not."""
    def denoise_fn(video_state, audio_state, sigmas, step_idx):
        sigma = float(sigmas[step_idx])
        blend = min(1.0, 1.2 * (1 - sigma))   # Predictions sharpen as noise falls
        pos_v = blend * v_target + (1 - blend) * video_state.latent
        pos_a = blend * a_target + (1 - blend) * audio_state.latent
        if not guider.enabled():
            return pos_v, pos_a
        neg_v, neg_a = 0.9 * pos_v, 0.9 * pos_a   # Unprompted drifts toward blandness
        # Apply CFG: output = pos + (scale-1)*(pos - neg)
        return guider.apply(pos_v, neg_v), guider.apply(pos_a, neg_a)
    return denoise_fn

stepper = EulerDiffusionStep()

# ---- Stage 1: 40 CFG-guided steps at "half resolution" (our sequence lengths) ----
stage_1_sigmas = torch.linspace(1.0, 0.0, 41)   # LTX2Scheduler shapes this curve in production
video_state = LatentState(latent=torch.randn(1, video_len, 1))
audio_state = LatentState(latent=torch.randn(1, audio_len, 1))
video_state, audio_state, history_1 = euler_denoising_loop(
    stage_1_sigmas, video_state, audio_state, stepper,
    make_toy_denoiser(video_target, audio_target, CFGGuider(scale=4.0)))
print("Stage 1 done: 40 steps with CFG scale 4.0")

# ---- Spatial upsampling between stages (learned 2x upsampler in production) ----
upsampled_video = F.interpolate(video_state.latent.permute(0, 2, 1),
                                scale_factor=2, mode="linear").permute(0, 2, 1)
video_target_hr = F.interpolate(video_target.permute(0, 2, 1),
                                scale_factor=2, mode="linear").permute(0, 2, 1)
print(f"Upsampled video latent: {video_len} -> {upsampled_video.shape[1]} tokens")

# ---- Stage 2: 4 distilled steps, no CFG, starting from the upsampled latent ----
STAGE_2_DISTILLED_SIGMA_VALUES = [0.909375, 0.725, 0.421875, 0.0]
distilled_sigmas = torch.tensor(STAGE_2_DISTILLED_SIGMA_VALUES)

# Re-noise to the first distilled sigma, as the pipeline does
noise_scale = distilled_sigmas[0]
video_state_2 = LatentState(latent=(1 - noise_scale) * upsampled_video
                                   + noise_scale * torch.randn_like(upsampled_video))
audio_state_2 = LatentState(latent=(1 - noise_scale) * audio_state.latent
                                   + noise_scale * torch.randn_like(audio_state.latent))

video_state_2, audio_state_2, history_2 = euler_denoising_loop(
    distilled_sigmas, video_state_2, audio_state_2, stepper,
    make_toy_denoiser(video_target_hr, audio_target, CFGGuider(scale=1.0)))  # No CFG
print("Stage 2 done: 3 refinement steps over 4 distilled sigmas, no CFG")

v_err = (video_state_2.latent - video_target_hr).abs().mean().item()
a_err = (audio_state_2.latent - audio_target).abs().mean().item()
print(f"Final mean error: video {v_err:.4f}, audio {a_err:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

steps_to_show = [0, 10, 25, 39]
for s in steps_to_show:
    axes[0].plot(history_1[s][0][0, :, 0], alpha=0.3 + 0.7 * s / 39, color="tab:blue",
                 label=f"step {s + 1}")
axes[0].plot(video_target[0, :, 0], "k--", label="target")
axes[0].set_title("Stage 1: video latent denoising")
axes[0].legend(fontsize=7)

axes[1].plot(video_state_2.latent[0, :, 0], color="tab:blue", label="video (final)")
axes[1].plot(video_target_hr[0, :, 0], "k--", label="video target")
axes[1].plot(np.linspace(0, video_target_hr.shape[1] - 1, audio_len),
             audio_state_2.latent[0, :, 0], color="tab:orange", label="audio (final)")
axes[1].set_title("After Stage 2: events aligned across modalities")
axes[1].legend(fontsize=7)

axes[2].plot(stage_1_sigmas, label="Stage 1 (40 steps, CFG)", marker=".", markersize=3)
axes[2].plot(np.linspace(0, 40, 4), distilled_sigmas, label="Stage 2 (4 sigmas, no CFG)",
             marker="o")
axes[2].set_xlabel("Step")
axes[2].set_ylabel("Sigma")
axes[2].set_title("Sigma schedules")
axes[2].legend(fontsize=7)
plt.tight_layout()
plt.show()

### The frame-count constraint

LTX-2 requires `frames % 8 == 1` because of the video VAE's 8x temporal compression. A small validator saves a failed generation run.

In [ ]:
def validate_num_frames(num_frames: int) -> int:
    """LTX-2 constraint: frames % 8 == 1 (valid: 1, 9, 17, 25, ..., 121)."""
    if num_frames < 1 or num_frames % 8 != 1:
        lower = max(1, ((num_frames - 1) // 8) * 8 + 1)
        raise ValueError(f"num_frames must satisfy frames % 8 == 1; got {num_frames}. "
                         f"Nearest valid values: {lower} or {lower + 8}.")
    return num_frames

print("121 frames:", validate_num_frames(121), "(valid)")
try:
    validate_num_frames(120)
except ValueError as error:
    print("120 frames:", error)

## Hands-On Implementation: Running the Real LTX-2

Everything above ran on toy scales. Running the real model needs serious hardware (24 GB VRAM minimum with FP8, 48 GB+ for BF16 at full resolution) and roughly 40 GB of downloads, so this section is opt-in: flip the flag only on a machine that can handle it.

In [ ]:
# Opt-in flag: set to True ONLY on a machine with a 24 GB+ GPU and ~60 GB free disk.
RUN_LTX2_PIPELINE = False

# Required components (download before enabling):
# 1. Main checkpoint:      https://huggingface.co/Lightricks/LTX-2
#    ltx-2-19b-dev.safetensors (~38 GB) or ltx-2-19b-dev-fp8.safetensors (~19 GB)
# 2. Spatial upsampler:    ltx2_spatial_upscaler.safetensors
# 3. Distilled LoRA:       ltx2_distilled_lora.safetensors
# 4. Gemma 3 text encoder: https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized

# Environment setup (run in a terminal):
#   git clone https://github.com/Lightricks/LTX-2
#   cd LTX-2
#   uv sync --frozen
#   source .venv/bin/activate

print("RUN_LTX2_PIPELINE =", RUN_LTX2_PIPELINE)

### Text-to-video with the two-stage pipeline

The production entry point is `TI2VidTwoStagesPipeline`. Note how the call mirrors what we built: 40 Stage 1 steps, CFG scale 4.0, the distilled LoRA for Stage 2, and a frame count satisfying the modulus constraint.

In [ ]:
if RUN_LTX2_PIPELINE:
    from ltx_core.loader import LTXV_LORA_COMFY_RENAMING_MAP, LoraPathStrengthAndSDOps
    from ltx_pipelines.ti2vid_two_stages import TI2VidTwoStagesPipeline

    # Configure distilled LoRA for Stage 2
    distilled_lora = [
        LoraPathStrengthAndSDOps(
            path="/path/to/ltx2_distilled_lora.safetensors",
            strength=0.6,
            sd_ops=LTXV_LORA_COMFY_RENAMING_MAP,
        ),
    ]

    # Initialize the two-stage pipeline
    pipeline = TI2VidTwoStagesPipeline(
        checkpoint_path="/path/to/ltx-2-19b-dev.safetensors",
        distilled_lora=distilled_lora,
        spatial_upsampler_path="/path/to/ltx2_spatial_upscaler.safetensors",
        gemma_root="/path/to/gemma-3-12b",
        loras=[],                # Optional: additional custom LoRAs
        fp8transformer=True,     # Enable FP8 for lower VRAM
    )

    # Generate video with audio
    pipeline(
        prompt="A serene mountain lake at sunrise, with mist rising from the water "
               "and birds singing",
        output_path="output.mp4",
        seed=42,
        height=1024,             # Final output height (Stage 1 uses 512)
        width=1536,              # Final output width (Stage 1 uses 768)
        num_frames=121,          # Must satisfy frames % 8 == 1
        frame_rate=25.0,
        num_inference_steps=40,  # Stage 1 steps (Stage 2 uses 4)
        cfg_guidance_scale=4.0,
    )
else:
    print("Skipped: set RUN_LTX2_PIPELINE = True to run the full model.")

### Command-line interface

Every pipeline also runs directly from the shell. The distilled pipeline trades some quality for speed (8 steps total).

In [ ]:
# Two-stage text-to-video (recommended for production):
#
#   python -m ltx_pipelines.ti2vid_two_stages \
#       --checkpoint-path /path/to/ltx-2-19b-dev.safetensors \
#       --distilled-lora /path/to/ltx2_distilled_lora.safetensors 0.6 \
#       --spatial-upsampler-path /path/to/ltx2_spatial_upscaler.safetensors \
#       --gemma-root /path/to/gemma-3-12b \
#       --prompt "A beautiful sunset over the ocean" \
#       --output-path output.mp4 \
#       --height 1024 --width 1536 \
#       --num-frames 121
#
# Enable FP8 for lower memory usage:
#   PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
#   python -m ltx_pipelines.ti2vid_two_stages --enable-fp8 ...
#
# Fast distilled pipeline (8 steps total):
#   python -m ltx_pipelines.distilled ...
#
# View all options:
#   python -m ltx_pipelines.ti2vid_two_stages --help
print("CLI reference cell: copy commands into a terminal on your GPU machine.")

### Image-to-video and speech synthesis

Conditioning on keyframes animates still images, and `<S>...<E>` tags in the prompt produce synchronized speech with matching lip movements.

In [ ]:
if RUN_LTX2_PIPELINE:
    # Image conditioning: specify (image_path, frame_index, strength)
    pipeline(
        prompt="A cat slowly turning its head and blinking",
        output_path="animated_cat.mp4",
        seed=42,
        height=1024, width=1536,
        num_frames=49,
        frame_rate=25.0,
        images=[("cat_photo.jpg", 0, 1.0)],   # Condition at frame 0, full strength
    )

    # Speech tags: <S>...<E> marks spoken content
    speech_prompt = (
        "A news anchor sitting at a desk, speaking clearly to the camera. "
        "<S>Good evening. Today we bring you breaking news from around the world. "
        "Scientists have made a remarkable discovery that could change everything "
        "we know about renewable energy.<E>"
    )
    pipeline(
        prompt=speech_prompt,
        output_path="news_anchor.mp4",
        seed=42,
        height=1024, width=1536,
        num_frames=121,
        frame_rate=25.0,
    )
    # The generated audio contains synchronized speech matching the lip movements.
else:
    print("Skipped: set RUN_LTX2_PIPELINE = True to run these examples.")

### Fine-tuning with LoRA

The `ltx-trainer` package covers dataset prep, configuration, and training. The `with_audio: true` flag is what makes a LoRA learn synchronized audio-video behavior rather than video alone.

In [ ]:
# Step 1: Organize videos with captions as JSON:
#   [{"video": "video1.mp4", "caption": "A dog playing in the park"},
#    {"video": "video2.mp4", "caption": "Ocean waves at sunset"}]
#
# Step 2 (optional): auto-caption videos:
#   uv run python scripts/caption_videos.py videos_dir/ --output dataset.json
#
# Step 3: precompute latents and embeddings:
#   uv run python scripts/process_dataset.py dataset.json \
#       --resolution-buckets "960x544x49" \
#       --model-path /path/to/ltx-2-19b-dev.safetensors \
#       --text-encoder-path /path/to/gemma-3-12b

training_config = """
# configs/my_training.yaml
model:
  model_path: "/path/to/ltx-2-19b-dev.safetensors"
  text_encoder_path: "/path/to/gemma-3-12b"
  training_mode: "lora"

training_strategy:
  name: "text_to_video"
  first_frame_conditioning_p: 0.1
  with_audio: true                  # Enable audio training
  audio_latents_dir: "audio_latents"

lora:
  rank: 64
  target_modules:
    - "to_k"   # Matches video, audio, and cross-modal attention
    - "to_q"
    - "to_v"
    - "to_out.0"

data:
  preprocessed_data_root: "/path/to/preprocessed/data"

optimization:
  learning_rate: 1e-4
  train_batch_size: 1
  gradient_accumulation_steps: 4
  max_train_steps: 1000

output_dir: "outputs/my_lora"
"""
print(training_config)

# Launch training:
#   uv run python scripts/train.py configs/my_training.yaml              # Single GPU
#   uv run accelerate launch scripts/train.py configs/my_training.yaml   # Multi-GPU

In [ ]:
if RUN_LTX2_PIPELINE:
    # Use a trained LoRA for inference by adding it to the pipeline's loras list
    custom_loras = [
        LoraPathStrengthAndSDOps(
            path="/path/to/my_trained_lora.safetensors",
            strength=0.8,
            sd_ops=LTXV_LORA_COMFY_RENAMING_MAP,
        ),
    ]
    pipeline_with_lora = TI2VidTwoStagesPipeline(
        checkpoint_path="/path/to/ltx-2-19b-dev.safetensors",
        distilled_lora=distilled_lora,
        spatial_upsampler_path="/path/to/ltx2_spatial_upscaler.safetensors",
        gemma_root="/path/to/gemma-3-12b",
        loras=custom_loras,          # Your trained LoRA
    )
    pipeline_with_lora(prompt="Your custom style applied to a new scene",
                       output_path="custom_style.mp4", seed=42,
                       height=1024, width=1536, num_frames=121, frame_rate=25.0)
else:
    print("Skipped: set RUN_LTX2_PIPELINE = True to run LoRA inference.")

## Summary

You built the core ideas of synchronized audio-video generation at runnable scale:

- A unified transformer processes audio and video together, so alignment emerges during generation rather than being bolted on afterward.
- Bidirectional cross-modal attention at every layer lets each modality shape the other continuously.
- Audio lives in a structured latent space: log-mel spectrograms, per-channel normalized VAE latents with 4x temporal compression, and timecoded tokens.
- A HiFi-GAN style vocoder reconstructs 24 kHz waveforms with 240x upsampling and multi-receptive-field fusion.
- Two-stage inference separates structure (40 CFG-guided steps at half resolution) from detail (4 distilled steps at full resolution).

Chapter 8 turns these building blocks into production workflows with ComfyUI.

*Video Generation with AI* by Joseph Enochs, published by O'Reilly Media.